# Responses API: Concurrent MCP Scenario

This notebook mirrors `release_room.scenarios.concurrent_security_alert_enrichment`. It teaches the `concurrent-security-alert-enrichment` scenario in small executable blocks, with an added focus on **MCP tool usage**: scenario metadata, pattern anatomy, the flow diagram, the local MCP tool context, agent roles, the Responses-vs-Invocations API fit, workflow execution, and the matching Responses API shape.

## 1. Notebook Setup

Run this notebook from the project virtual environment after installing the package with `python -m pip install -e .`. The MCP tools run locally over stdio with no network access and no credentials.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start):
    current = Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "release_room").exists():
            return candidate
        nested = candidate / "responses-api-release-room"
        if (nested / "src" / "release_room").exists():
            return nested
    raise RuntimeError("Could not find responses-api-release-room project root.")

PROJECT_ROOT = find_project_root(Path.cwd())
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

PROJECT_ROOT

## 2. Load The Scenario And Helpers

Each scenario has its own Python module with a single `SCENARIO` object. Notebook helpers keep repeated learning/display code out of the notebook cells.

In [ ]:
from release_room.scenarios.concurrent_security_alert_enrichment import SCENARIO
from release_room.scenarios import SCENARIOS_BY_ID, get_scenario
from release_room.notebook_helpers import (
    agent_roster,
    default_ollama_kwargs,
    mcp_tool_context,
    pattern_anatomy,
    responses_api_reference,
    scenario_summary,
    workflow_result_to_text,
)

scenario = SCENARIO
assert SCENARIOS_BY_ID["concurrent-security-alert-enrichment"] is scenario
assert get_scenario("concurrent-security-alert-enrichment") is scenario

scenario_summary(scenario)

## 3. Pattern Anatomy

This is the orchestration behavior. The Responses API boundary stays stable while the server runs this pattern and the agents call local MCP tools underneath.

In [ ]:
pattern_anatomy(scenario)

## 4. Flow Diagram

This runtime Mermaid diagram shows the API boundary, orchestration pattern, the scenario agents, and **dashed links to each agent's MCP tools**. The helper returns the Mermaid source and displays an image.

In [ ]:
from release_room.diagram_helpers import display_scenario_flow, scenario_flow_diagram

flow_diagram = display_scenario_flow(scenario)
flow_diagram.mermaid

## 5. MCP Tool Context

These agents are grounded by the bundled `enterprise-context` MCP server. It is a local FastMCP stdio server with embedded sample enterprise data: **no network, no credentials, no writes**. The cells below list the tools the server exposes, show which tools each agent is allowed to call, and call every tool directly through the helper layer so you can see the deterministic results before the agents use them.

In [ ]:
from release_room.mcp_servers import enterprise_context

context = mcp_tool_context(scenario)
print("Server exposes:", enterprise_context.AVAILABLE_TOOLS)
print("Tools used in this scenario:", context["all_tools_used"])
context["tools_by_agent"]

In [ ]:
# Call every local MCP tool directly. These are plain in-process calls into the same
# functions the stdio server exposes, so they need no Ollama and no subprocess.
record = enterprise_context.lookup_enterprise_record("ALERT-2298")
policies = enterprise_context.search_policy("identity token incident")
steps = enterprise_context.list_playbook_steps("security-enrichment")
score = enterprise_context.calculate_priority_score(impact=4, urgency=4, scope=3)
decision = enterprise_context.create_decision_log_entry("ALERT-2298 review", "approve")
{
    "record": record,
    "policy_matches": policies["match_count"],
    "playbook_steps": steps["step_count"],
    "priority": score,
    "decision_log": decision,
}

## 6. Agent Roster

The roster shows who participates, what each agent is instructed to do, and (via the MCP context above) which tools each agent may call.

In [ ]:
agent_roster(scenario)

## 7. Configure Ollama

The default notebook settings are conservative so local multi-agent runs finish predictably. Make sure Ollama is running and `qwen3:14b` is pulled.

In [ ]:
from release_room.agents import build_ollama_config

ollama_kwargs = default_ollama_kwargs()
config = build_ollama_config(**ollama_kwargs)
config

## 8. API Fit: Responses vs Invocations

This MCP scenario is mirrored in both sample projects, so it is a good place to make the API choice explicit.

- **Use the Responses API** when the caller is a chat-style client that sends OpenAI-style `input` and expects a conversational reply. The server picks one scenario at startup and every turn flows through the same orchestration. The MCP tools run server-side and stay invisible to the caller's contract.
- **Use the Invocations API** when the caller is a system (webhook, CI job, batch worker) that sends a structured job payload and expects an application-owned JSON response with metadata. The scenario is selected per request, which makes it easy to route different job types to different orchestration patterns.

In both cases the same local `enterprise-context` MCP server provides deterministic, credential-free tool grounding, so the orchestration logic is identical and only the API boundary differs.

## 9. Build And Run The Workflow

This uses the same builder path as the `/responses` server, but runs in-process. When the agents run, the framework launches the local MCP stdio server with `approval_mode="never_require"` and exposes only each agent's allowed tools.

In [ ]:
from release_room.workflows import build_release_workflow

workflow = build_release_workflow(
    "concurrent-security-alert-enrichment",
    model=config.model,
    ollama_host=config.host,
    temperature=config.temperature,
    num_ctx=config.num_ctx,
    max_tokens=config.max_tokens,
    keep_alive=config.keep_alive,
    think=config.think,
)
workflow

In [ ]:
result = await workflow.run(scenario.sample_input)
print(workflow_result_to_text(result)[:5000])

## 10. Responses API Boundary

Use this reference to compare the in-process workflow with the hosted Responses endpoint. The MCP tools stay entirely server-side, so the API contract is unchanged.

In [ ]:
responses_api_reference(scenario)

## 11. Experiments

Try a different record id in the MCP tool cell, change one agent's `mcp_tools`, adjust `max_tokens` or `temperature`, or edit an agent instruction in the scenario module. Rebuild the workflow after changing scenario code.